The following implementation demonstrates a parellel processing workflow constructed with the LangChain framework.

This workflow is designed to execute two independent operations concurrently in response to a single user query. These parallel processes are instantiates as distinct chains or functions, and theri respective outputs are subsequently aggregated into a unified result.

In [1]:
import os
import asyncio
from typing import Optional

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough

In [3]:
try:
    llm: Optional[ChatOllama] = ChatOllama(model="llama3", temperature=0.7)
except Exception as e:
    print(f"Error initializing language model: {e}")

# --- Define Independent Chains ---
# These three chains represent distinct tasks that can be executed in parallel.

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize the following topic concisely:"),
        ("user", "{topic}™")
    ])
    | llm
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three interesting questions about the following topic:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Identify 5-10 key terms from the following topic, separeated by commas:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

# --- Build the Parallel + Synthesis Chhain ---

# 1. Define the block of tasks to run in parallel. The results of these, along with the original topic, will be fed into the next step.
map_chain = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(), # Pass the original topic through
    }
)
# 2. Define the final synthesis prompt which will combine the parallel results.
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
    Summary: {summary}
    Related Questions: {questions}
    Key Terms: {key_terms}
    Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}")
])
# 3. Construct the full chain by piping the parallel results directly into the synthesis prompt, followed by he LLM and output parser.
full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()

In [6]:
# --- Run the Chain ---
async def run_parallel_example(topic: str) -> None:
    """
    Asynchronously invokes the parallel processing chain with a specific topic
    and prints the synthesized result.

    Args:
        topic: The input topic to be processed by the LangChain chains.
    """
    if not llm:
        print("LLM not initialized. Cannot run example.")
        return

    print(f"\n--- Running Parallel LangChain Example for topic: '{topic}' ---")
    try:
        # the input to `ainvoke` is the single 'topic' string, then passed to each runnable in the `map_chain`.
        response = await full_parallel_chain.ainvoke(topic)
        print("\n--- Final Response ---")
        print(response)
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")

if __name__ == "__main__":
    test_topic = "The history of space exploration"
    await run_parallel_example(test_topic)


--- Running Parallel LangChain Example for topic: 'The history of space exploration ---

--- Final Response ---
The history of space exploration is a rich and fascinating story that spans centuries, from the early astronomers who gazed up at the stars to the modern-day missions that are currently exploring our solar system and beyond.

**Early Years (16th-19th centuries)**

The earliest recorded interest in astronomy and celestial bodies dates back to the 16th century, with pioneers like Galileo Galilei observing the Moon with a telescope in 1609. Johannes Kepler's discovery of laws of planetary motion between 1609-1619 laid the foundation for modern astrophysics.

**Space Age Begins (1950s-1960s)**

The Soviet Union launched Sputnik 1, the world's first artificial satellite, in 1957, marking the beginning of the Space Age. The United States responded with Explorer 1, its first satellite, just a year later. The Mercury program sent astronauts to space for the first time in 1959-1963, 